In [ ]:
from dotenv import load_dotenv

load_dotenv()


In [ ]:
TEN_CONG_TY = "ABC",

prompt = """
Bạn là chuyên gia phân tích dữ liệu doanh thu.

Hãy tạo báo cáo doanh thu theo định dạng **Markdown** với các yêu cầu sau:

1. Tiêu đề:
   - "Doanh thu Công ty {TEN_CONG_TY} năm {NAM}"

2. Bảng dữ liệu markdown gồm các cột:
   - Chỉ tiêu
   - Nguồn CRM
   - Nguồn Kinh doanh

3. Các dòng chỉ tiêu bắt buộc:
   - Tổng doanh thu thực tế
   - Tổng doanh thu theo đơn vị tiền VND
   - Tổng doanh thu theo đơn vị USD

4. Dữ liệu đầu vào:
   - Doanh thu từ CRM: {DOANH_THU_CRM_VND}, {DOANH_THU_CRM_USD}
   - Doanh thu từ Kinh doanh: {DOANH_THU_KD_VND}, {DOANH_THU_KD_USD}
   - Doanh thu thực tế CRM: {DOANH_THU_THUC_TE_CRM}
   - Doanh thu thực tế Kinh doanh: {DOANH_THU_THUC_TE_KD}

5. Nhận xét:
   - Viết 1–2 câu nhận xét ngắn gọn
   - So sánh số liệu giữa nguồn CRM và Kinh doanh
   - Nêu khả năng chênh lệch do khác nhau về nguồn hoặc cách ghi nhận

6. Ràng buộc định dạng:
   - Chỉ trả về **Markdown**
   - Không giải thích thêm
   - Không dùng bullet ngoài bảng
   - Nhận xét đặt ngay dưới bảng, in đậm tiêu đề "Nhận xét:"

Ví dụ đầu ra mong muốn:
(Chỉ dùng làm tham chiếu, không lặp lại số liệu mẫu)

"""

In [ ]:
from openai import OpenAI
import json
json.dumps()
client = OpenAI(
    api_key=os.getenv("VCS_LLM_API_KEY"),
    base_url=os.getenv("VCS_LLM_API_KEY")
)

In [ ]:
data = {
    "company": "ABC",
    "year": 2025,
    "revenue": {
        "crm": {
            "actual": "2.450 tỷ VND",
            "vnd": 2450000000000,
            "usd": 98000000
        },
        "business": {
            "actual": "2.500 tỷ VND",
            "vnd": 2500000000000,
            "usd": 100000000
        }
    }
}



def build_prompt(data: dict, user_query: str) -> str:
    data_json = json.dumps(data, ensure_ascii=False, indent=2)
    return f"""
Dưới đây là dữ liệu doanh thu đã được chuẩn hóa:

DATA_JSON:
{data_json}

YÊU CẦU NGƯỜI DÙNG:
{user_query}

Hãy tạo báo cáo theo các quy tắc sau:

1. Định dạng đầu ra: Markdown
2. Tiêu đề: "Doanh thu Công ty {data['company']} năm {data['year']}"
3. Bảng markdown gồm các cột:
   - Chỉ tiêu
   - Nguồn CRM
   - Nguồn Kinh doanh
4. Các dòng chỉ tiêu bắt buộc:
   - Tổng doanh thu thực tế
   - Tổng doanh thu theo đơn vị tiền VND
   - Tổng doanh thu theo đơn vị USD
5. Viết phần **Nhận xét** ngay sau bảng:
   - So sánh số liệu giữa CRM và Kinh doanh
   - Nêu khả năng chênh lệch dữ liệu
6. Không giải thích, không thêm nội dung ngoài Markdown

CHỈ TRẢ VỀ MARKDOWN.
"""


def generate_markdown_report(data: dict, user_query: str) -> str:
    prompt = build_prompt(data, user_query)
    response = client.chat.completions.create(
            model=None,
            messages=[
                {"role": "system", "content": "Bạn là chuyên gia BI & Data Analyst."},
                {"role": "user", "content": prompt}
            ],
            temperature=0.2
        )

    return response.choices[0].message.content.strip()


In [4]:
user_query = "tạo báo cáo doanh thu công ty ABC năm 2025, so sánh CRM và Kinh doanh"

markdown_output = generate_markdown_report(data, user_query)
print(markdown_output)

# Doanh thu Công ty ABC năm 2025

| Chỉ tiêu                         | Nguồn CRM                | Nguồn Kinh doanh          |
|----------------------------------|--------------------------|---------------------------|
| Tổng doanh thu thực tế           | 2.450 tỷ VND             | 2.500 tỷ VND              |
| Tổng doanh thu theo đơn vị tiền VND | 2,450,000,000,000 VND   | 2,500,000,000,000 VND     |
| Tổng doanh thu theo đơn vị USD   | 98,000,000 USD           | 100,000,000 USD           |

**Nhận xét**  
- Doanh thu thực tế của nguồn Kinh doanh cao hơn CRM khoảng 50 tỷ VND (≈ 2%); tương ứng, doanh thu tính bằng VND và USD cũng lớn hơn.  
- Khoảng chênh lệch có thể xuất phát từ các yếu tố như thời gian ghi nhận, tỷ giá hối đoái áp dụng hoặc các khoản doanh thu chưa được đồng bộ giữa hai nguồn.
